# A training loop: hands-on (optional)
**LLMs, Animated · Lesson P4.2**

You'll write the five-line PyTorch training loop yourself, twice:
1. fitting a line with `nn.Linear`, `nn.MSELoss` and SGD,
2. training a small classifier with a `DataLoader`, `nn.CrossEntropyLoss`, AdamW, a validation set and train/eval mode.

Everything runs on a CPU in a few seconds. Each exercise has a `# TODO` cell and a **check** cell; solutions are at the bottom.

In [ ]:
import math
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(0)
print("torch", torch.__version__)

## 1 · The five lines
Data from $y = 2x + 1$ plus noise. The model is `nn.Linear(1, 1)`, which holds a weight $w$ and a bias $b$.
Complete `train_step`: **zero_grad → forward → loss → backward → step**, and return the loss as a Python float.

In [ ]:
X = torch.rand(64, 1) * 4 - 2
Y = 2 * X + 1 + 0.1 * torch.randn(64, 1)

In [ ]:
def train_step(model, opt, loss_fn, xb, yb):
    # TODO: the five lines; return loss.item()
    raise NotImplementedError("Exercise 1")

**Check 1.** Full-batch training for 200 steps.

In [ ]:
model = nn.Linear(1, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()
losses = [train_step(model, opt, loss_fn, X, Y) for _ in range(200)]
w, b = model.weight.item(), model.bias.item()
print(f"w = {w:.3f}, b = {b:.3f}, loss {losses[0]:.3f} → {losses[-1]:.4f}")
assert isinstance(losses[0], float)
assert abs(w - 2) < 0.05 and abs(b - 1) < 0.05
print("✓ exercise 1")

## 2 · A model as an `nn.Module`
Three clusters of 2-D points, one per class. Define `MLP`: `Linear(2, 32) → ReLU → Dropout(0.1) → Linear(32, 3)`.
`forward` must return **logits** (raw scores), shape `(B, 3)`. No softmax: `nn.CrossEntropyLoss` applies it internally.

In [ ]:
def make_blobs(n, seed):
    g = torch.Generator().manual_seed(seed)
    y = torch.randint(0, 3, (n,), generator=g)
    centers = torch.tensor([[0., 2.], [-2., -1.], [2., -1.]])
    X = centers[y] + 0.8 * torch.randn(n, 2, generator=g)
    return X, y

X_train, y_train = make_blobs(1000, seed=1)
X_val, y_val = make_blobs(300, seed=2)
print(X_train.shape, y_train.shape, y_train[:10])

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: self.net = nn.Sequential(...)
        raise NotImplementedError("Exercise 2")

    def forward(self, x):
        # TODO
        raise NotImplementedError("Exercise 2")

**Check 2**

In [ ]:
m = MLP()
n_params = sum(p.numel() for p in m.parameters())
assert n_params == 2 * 32 + 32 + 32 * 3 + 3, n_params
assert m(X_train[:5]).shape == (5, 3)
m.eval()
with torch.no_grad():
    assert torch.equal(m(X_train[:5]), m(X_train[:5])), "in eval mode dropout is off, so outputs are deterministic"
print("✓ exercise 2:", n_params, "parameters")

## 3 · Evaluate properly
Write `evaluate(model, X, y)` returning `(loss, accuracy)` as Python floats. It must:
- switch to **eval mode** (dropout off),
- run under **`torch.no_grad()`** (no graph),
- compute the mean cross-entropy and the fraction of correct `argmax` predictions.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def evaluate(model, X, y):
    # TODO
    raise NotImplementedError("Exercise 3")

**Check 3.** An untrained model's loss should be in the neighbourhood of $\ln 3 \approx 1.10$ (P2.1), the loss of a uniform guess. Its accuracy is whatever its random weights happen to give (it can even be 0% if the random weights map every cluster to a wrong label).

In [ ]:
torch.manual_seed(0)
m = MLP()
l, a = evaluate(m, X_val, y_val)
print(f"untrained: loss {l:.3f}, acc {a:.2f}")
assert isinstance(l, float) and isinstance(a, float)
assert not m.training, "evaluate should leave the model in eval mode"
assert 0.6 < l < 2.0 and 0.0 <= a <= 1.0
print("✓ exercise 3")

## 4 · The full loop
Put it together: a `DataLoader` with `batch_size=64, shuffle=True`, AdamW with `lr=1e-2`, and for each epoch:
**train mode** → loop over batches with your `train_step` → `evaluate` on the validation set → append to `history`.

In [ ]:
def train(epochs=10, lr=1e-2, seed=0):
    torch.manual_seed(seed)
    model = MLP()
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    # TODO: build train_loader from X_train, y_train
    history = []   # one (train_loss_avg, val_loss, val_acc) tuple per epoch
    for epoch in range(epochs):
        # TODO: model.train(); loop over batches with train_step; average the batch losses
        # TODO: val_loss, val_acc = evaluate(...); history.append(...)
        raise NotImplementedError("Exercise 4")
    return model, history

**Check 4**

In [ ]:
model, history = train()
for e, (tl, vl, va) in enumerate(history):
    print(f"epoch {e}: train loss {tl:.3f} | val loss {vl:.3f} | val acc {va:.1%}")
assert len(history) == 10
assert history[-1][1] < history[0][1], "validation loss should improve"
assert history[-1][2] > 0.9, "a 3-blob problem should reach > 90% validation accuracy"
print("✓ exercise 4")

## Experiment
1. **Forget `zero_grad`.** Copy `train_step`, delete `opt.zero_grad()`, and train with it. What happens to the loss? Why?
2. **Learning rate.** Try `train(lr=1.0)` and `train(lr=1e-4)`. Which one diverges or bounces, which one crawls? (P2.2)
3. **Eval mode matters.** In `evaluate`, remove `model.eval()` and evaluate the same model twice. Do you get the same loss? Why not?
4. **An LLM's loss uses the same function.** Logits of shape `(B, T, V)` and targets `(B, T)` are flattened: run the cell below.

In [ ]:
B, T, V = 2, 5, 11
logits = torch.randn(B, T, V)
targets = torch.randint(0, V, (B, T))
print("next-token loss:", loss_fn(logits.reshape(B * T, V), targets.reshape(B * T)).item(),
      "| uniform guess would be ln V =", round(math.log(V), 3))

## Recap
- `nn.Module`: layers in `__init__`, data flow in `forward`; call it as `model(x)`.
- The loop: **zero_grad → forward → loss → backward → step**, once per mini-batch.
- `DataLoader` gives shuffled mini-batches; one pass = one epoch.
- Evaluate with `model.eval()` **and** `torch.no_grad()`; log Python numbers with `.item()`.

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def train_step(model, opt, loss_fn, xb, yb):
    opt.zero_grad()
    pred = model(xb)
    loss = loss_fn(pred, yb)
    loss.backward()
    opt.step()
    return loss.item()

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Dropout(p=0.1),
            nn.Linear(32, 3),
        )

    def forward(self, x):
        return self.net(x)

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
loss_fn = nn.CrossEntropyLoss()

def evaluate(model, X, y):
    model.eval()
    with torch.no_grad():
        logits = model(X)
        loss = loss_fn(logits, y).item()
        acc = (logits.argmax(dim=1) == y).float().mean().item()
    return loss, acc

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def train(epochs=10, lr=1e-2, seed=0):
    torch.manual_seed(seed)
    model = MLP()
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
    history = []
    for epoch in range(epochs):
        model.train()
        batch_losses = [train_step(model, opt, loss_fn, xb, yb) for xb, yb in train_loader]
        val_loss, val_acc = evaluate(model, X_val, y_val)
        history.append((sum(batch_losses) / len(batch_losses), val_loss, val_acc))
    return model, history